In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import os

# DIRECT LSTM — GENERATION PIPELINE

device_torch = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEQ_LEN = 20

In [2]:
import os
import sys
import numpy as np
import pandas as pd
import sumolib

# SUMO MODULE

sumo_home = os.path.expanduser('~/sumo')
sys.path.append(os.path.join(sumo_home, 'tools'))
os.environ['SUMO_HOME'] = sumo_home

SUMO_DIR = 'D:\Thesis V2X\Project on Git\Berlin V2X Implementation\sumo1'
net_file = os.path.join(SUMO_DIR, 'osm1.net.xml')

print("Loading SUMO network...")
net = sumolib.net.readNet(net_file)
print(f"  Network loaded: {len(net.getEdges())} edges")

Loading SUMO network...
  Network loaded: 6055 edges


In [3]:
BASE_INPUTS = [
    'hour', 'day_of_week',
    'device_pc1', 'device_pc2', 'device_pc3', 'device_pc4',
    'direction_uplink',
    'measured_qos_delay',
    'measurement', 'operator'
]

ALL_TARGETS = [
    'Latitude', 'Longitude', 'speed_kmh', 'sin_COG', 'cos_COG', 'Altitude',
    'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed',
    'Traffic Jam Factor', 'Traffic Distance',
    'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
    'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
    'PCell_Downlink_frequency', 'PCell_Band_Indicator',
    'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
    'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)',
    'datarate', 'jitter_log', 'Pos in Ref Round', 'target_datarate',
    'ping_ms'
]

ALL_AUTO_COLS = ALL_TARGETS.copy()

SNAP_RULES = {
    'PCell_Downlink_frequency': [125.0, 475.0, 1300.0, 1801.0, 2850.0, 3050.0, 3749.0, 9460.0],
    'PCell_freq_MHz': [700.0, 900.0, 1800.0, 2000.0, 2100.0, 2600.0],
    'PCell_Band_Indicator': [1.0, 3.0, 7.0, 8.0, 28.0],
    'PCell_Downlink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Uplink_bandwidth_MHz': [5.0, 10.0, 15.0, 20.0],
    'PCell_Downlink_Average_MCS': list(range(0, 30)),
}

LOG_FEATURES = [
    'datarate', 'target_datarate',
    'PCell_Downlink_TB_Size', 'PCell_Uplink_TB_Size',
    'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
    'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs',
    'ping_ms', 'Pos in Ref Round', 'Traffic Distance',
    'PCell_Uplink_Tx_Power_(dBm)',
]

def snap_to_nearest(value, valid_set):
    valid_arr = np.array(valid_set)
    return valid_arr[np.argmin(np.abs(valid_arr - value))]

In [4]:
class DirectLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim,
                            num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0,
                            batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, output_dim)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [5]:

MODEL_DIR = 'research\generation_output'

print("Loading Direct LSTM model and scalers...")
scalers = pickle.load(open(os.path.join(MODEL_DIR, 'direct_multioutput_scalers.pkl'), 'rb'))
input_scaler = scalers['input_scaler']
target_scaler = scalers['target_scaler']



model = DirectLSTM(input_dim=47, hidden_dim=256, output_dim=37, num_layers=2, dropout=0.20)
model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'direct_lstm_model.pt'), map_location=device_torch))
model.to(device_torch)
model.eval()
print(f"  Model loaded: {sum(p.numel() for p in model.parameters())} parameters")

# Load dataset
print("Loading dataset...")
df = pd.read_pickle('research\models\impute_result\df_complete_clean.pkl')
df = df.sort_values('timestamp').reset_index(drop=True)
df['sin_COG'] = np.sin(np.radians(df['COG']))
df['cos_COG'] = np.cos(np.radians(df['COG']))
df['jitter_log'] = np.log1p(df['jitter'])
print(f"  Dataset loaded: {len(df)} rows")

Loading Direct LSTM model and scalers...
  Model loaded: 876325 parameters
Loading dataset...
  Dataset loaded: 204942 rows


In [6]:
def generate_direct_lstm(device_id, start_timestamp, n_steps=5, mode='with_gps',
                         lat=None, lon=None, positions=None):
    """
    Parameters:
        device_id:       'pc1', 'pc2', 'pc3', or 'pc4'
        start_timestamp: pd.Timestamp
        n_steps:         number of timesteps to generate (1-10 recommended)
        mode:            'with_gps' (lat/lon provided)
        lat:             latitude (required if mode='with_gps')
        lon:             longitude (required if mode='with_gps')
        positions:       list of dicts with 'lat','lon' per step
    """
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    mask = device_data['timestamp'] < start_ts
    if mask.sum() < SEQ_LEN:
        print(f"Error: Need {SEQ_LEN} rows before {start_ts}, found {mask.sum()}")
        return None

    seed = device_data[mask].tail(SEQ_LEN).reset_index(drop=True)

    # If single lat/lon given, auto-generate positions along the road
    if positions is None and mode == 'with_gps' and lat is not None:
        last_seed = seed.iloc[-1]
        # Build a fake last_row with the provided lat/lon and seed's speed/COG
        last_row_fake = {
            'Latitude': lat,
            'Longitude': lon,
            'speed_kmh': last_seed['speed_kmh'],
            'COG': last_seed['COG'],
            'timestamp': start_ts - pd.Timedelta(seconds=1),
        }
        
        positions = [{'lat': lat, 'lon': lon}]  # first step = given position

    buffer = {col: list(seed[col].values) for col in ALL_AUTO_COLS}
    base_buffer = {col: list(seed[col].values) for col in BASE_INPUTS}

    generated_rows = []

    for step in range(n_steps):
        current_ts = start_ts + pd.Timedelta(seconds=step)

        current_base = {
            'hour': current_ts.hour,
            'day_of_week': current_ts.dayofweek + 1,
            'device_pc1': int(device_id == 'pc1'),
            'device_pc2': int(device_id == 'pc2'),
            'device_pc3': int(device_id == 'pc3'),
            'device_pc4': int(device_id == 'pc4'),
            #'direction_downlink': base_buffer['direction_downlink'][-1],
            'direction_uplink': base_buffer['direction_uplink'][-1],
            #'measured_qos_datarate': base_buffer['measured_qos_datarate'][-1],
            'measured_qos_delay': base_buffer['measured_qos_delay'][-1],
            'measurement': base_buffer['measurement'][-1],
            'operator': base_buffer['operator'][-1],
        }

        for col in ALL_AUTO_COLS:
            buffer[col].append(0.0)
        for col in BASE_INPUTS:
            base_buffer[col].append(current_base[col])

        base_seq = np.zeros((SEQ_LEN, len(BASE_INPUTS)), dtype=np.float32)
        auto_seq = np.zeros((SEQ_LEN, len(ALL_AUTO_COLS)), dtype=np.float32)

        for t in range(SEQ_LEN):
            idx = len(base_buffer['hour']) - SEQ_LEN + t
            for j, col in enumerate(BASE_INPUTS):
                base_seq[t, j] = base_buffer[col][idx]
            for j, col in enumerate(ALL_AUTO_COLS):
                auto_seq[t, j] = buffer[col][idx]

        auto_seq[-1, :] = 0.0

        step_lat = None
        step_lon = None
        if positions is not None and step < len(positions):
            step_lat = positions[step]['lat']
            step_lon = positions[step]['lon']

        if step_lat is not None:
            lat_idx = ALL_AUTO_COLS.index('Latitude')
            lon_idx = ALL_AUTO_COLS.index('Longitude')
            auto_seq[-1, lat_idx] = step_lat
            auto_seq[-1, lon_idx] = step_lon

        x = np.concatenate([base_seq, auto_seq], axis=1)
        x_scaled = input_scaler.transform(x.reshape(-1, x.shape[1])).reshape(1, SEQ_LEN, -1)

        with torch.no_grad():
            pred_scaled = model(torch.FloatTensor(x_scaled).to(device_torch)).cpu().numpy()
            pred = target_scaler.inverse_transform(pred_scaled)[0]

        result = {}
        for i, col in enumerate(ALL_TARGETS):
            val = pred[i]
            if col in SNAP_RULES:
                val = snap_to_nearest(val, SNAP_RULES[col])
            result[col] = val

        if step_lat is not None:
            result['Latitude'] = step_lat
            result['Longitude'] = step_lon

        result['COG'] = np.degrees(np.arctan2(result['sin_COG'], result['cos_COG'])) % 360
        result['jitter'] = np.expm1(result['jitter_log'])
        result['timestamp'] = current_ts

        for col in ALL_AUTO_COLS:
            buffer[col][-1] = result[col]

        for col in BASE_INPUTS:
            result[col] = current_base[col]

        generated_rows.append(result)

    return pd.DataFrame(generated_rows)

In [7]:
def format_output(gen_df):

    out = gen_df.copy()

    # Reverse log transforms
    LOG_FEATURES = [
        'datarate', 'target_datarate',
        'PCell_Downlink_TB_Size', 'PCell_Uplink_TB_Size',
        'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High',
        'PCell_Downlink_Num_RBs', 'PCell_Uplink_Num_RBs',
        'ping_ms', 'Pos in Ref Round', 'Traffic Distance',
        'PCell_Uplink_Tx_Power_(dBm)'
    ]

    for col in LOG_FEATURES:
        if col in out.columns:
            out[col] = np.expm1(out[col])

    # Decode one-hot: device
    if 'device_pc1' in out.columns:
        out['device'] = 'unknown'
        for d in ['pc1', 'pc2', 'pc3', 'pc4']:
            out.loc[out[f'device_{d}'] == 1, 'device'] = d
        out = out.drop(columns=['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4'], errors='ignore')
    
    # Decode one-hot: direction
    if 'direction_downlink' in out.columns:
        out['direction'] = 'unknown'
        #out.loc[out['direction_downlink'] == 1, 'direction'] = 'downlink'
        out.loc[out['direction_uplink'] == 1, 'direction'] = 'uplink'
        out = out.drop(columns=['direction_downlink', 'direction_uplink'], errors='ignore')

    # Decode one-hot: measured_qos
    if 'measured_qos_datarate' in out.columns:
        out['measured_qos'] = 'unknown'
        #out.loc[out['measured_qos_datarate'] == 1, 'measured_qos'] = 'datarate'
        out.loc[out['measured_qos_delay'] == 1, 'measured_qos'] = 'delay'
        out = out.drop(columns=['measured_qos_datarate', 'measured_qos_delay'], errors='ignore')

    # Drop internal columns
    drop_cols = ['sin_COG', 'cos_COG', 'jitter_log', 'hour', 'day_of_week']
    out = out.drop(columns=[c for c in drop_cols if c in out.columns], errors='ignore')

    # Round-off
    float_cols = out.select_dtypes(include=[np.number]).columns
    for col in float_cols:
        if col in ['Latitude', 'Longitude']:
            out[col] = out[col].round(7)
        elif col in ['PCell_freq_MHz', 'PCell_Downlink_frequency', 'PCell_Band_Indicator',
                     'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz',
                     'PCell_Downlink_Average_MCS', 'measurement', 'operator']:
            out[col] = out[col].round(0).astype(int)
        else:
            out[col] = out[col].round(2)

    first_cols = ['timestamp', 'device', 'direction', 'measured_qos']
    available_first = [c for c in first_cols if c in out.columns]
    remaining = [c for c in out.columns if c not in available_first]
    out = out[available_first + remaining]

    return out


In [8]:

# COMPARISON FUNCTION

def compare_with_real(device_id, start_timestamp, n_steps=5, mode='full',
                      lat=None, lon=None):
    
    
    start_ts = pd.Timestamp(start_timestamp)
    if start_ts.tzinfo is None:
        start_ts = start_ts.tz_localize('Europe/Berlin')

    device_col = f'device_{device_id}'
    device_data = df[df[device_col] == 1].sort_values('timestamp').reset_index(drop=True)

    real_mask = (device_data['timestamp'] >= start_ts) & \
                (device_data['timestamp'] < start_ts + pd.Timedelta(seconds=n_steps))
    real = device_data[real_mask].head(n_steps).reset_index(drop=True)

    gen = generate_direct_lstm(device_id, start_timestamp, n_steps, mode, lat, lon)
    if gen is None:
        return None, None

    n = min(len(gen), len(real))

    
    print(f" LSTM — {mode.upper()} | {device_id} | {start_timestamp} | {n_steps} steps")
    
    print(f"\n{'Feature':30s} | {'RMSE':>10s} | {'MAE':>10s} | {'Real (first 3)':>30s} | {'Gen (first 3)':>30s}")
    print("─" * 120)

    compare_cols = [
        ('Latitude', ''), ('Longitude', ''), ('speed_kmh', 'km/h'),
        ('COG', '°'), ('Altitude', 'm'), ('temperature', '°C'),
        ('PCell_RSRP_max', 'dBm'), ('PCell_freq_MHz', 'MHz'),
        ('PCell_Downlink_frequency', ''),
        ('datarate', '(log)'), ('ping_ms', '(log)'), ('jitter_log', '(log)'),
    ]

    for col, unit in compare_cols:
        if col not in real.columns or col not in gen.columns:
            continue
        rv = real[col].values[:n]
        gv = gen[col].values[:n]

        if col == 'COG':
            diff = np.abs(rv - gv)
            diff = np.minimum(diff, 360 - diff)
            rmse = np.sqrt(np.mean(diff ** 2))
            mae = np.mean(diff)
        else:
            rmse = np.sqrt(np.mean((rv - gv) ** 2))
            mae = np.mean(np.abs(rv - gv))

        r_str = ', '.join([f'{v:.4f}' for v in rv[:3]])
        g_str = ', '.join([f'{v:.4f}' for v in gv[:3]])
        print(f"{col + ' ' + unit:30s} | {rmse:10.4f} | {mae:10.4f} | {r_str:>30s} | {g_str:>30s}")

    return gen, real



In [9]:
# TEST

gen1, real1 = compare_with_real('pc2', '2021-06-23 14:12:51+02:00', n_steps=5,
                                 mode='with_gps', lat=52.504327, lon=13.335717)

 LSTM — WITH_GPS | pc2 | 2021-06-23 14:12:51+02:00 | 5 steps

Feature                        |       RMSE |        MAE |                 Real (first 3) |                  Gen (first 3)
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Latitude                       |     0.0011 |     0.0011 |      52.5042, 52.5042, 52.5041 |      52.5043, 52.5028, 52.5028
Longitude                      |     0.0044 |     0.0040 |      13.3356, 13.3355, 13.3355 |      13.3357, 13.3309, 13.3307
speed_kmh km/h                 |     1.3211 |     1.1411 |         2.6755, 2.9128, 2.9809 |         2.4679, 2.2217, 1.8677
COG °                          |     3.1281 |     2.7199 |   206.3000, 206.6000, 206.7000 |   207.4446, 207.9836, 208.6940
Altitude m                     |     0.7718 |     0.7367 |      32.6000, 32.6000, 32.5000 |      32.8835, 33.4166, 33.4191
temperature °C                 |     0.0789 |     0.0729 |      20.4100, 20.410

In [10]:
gen1=generate_direct_lstm('pc1', '2021-06-22 19:00:00+02:00', n_steps=5, mode='with_gps', lat=52.5134, lon=13.3341)
gen1

,Latitude,Longitude,speed_kmh,sin_COG,cos_COG,Altitude,precipIntensity,precipProbability,temperature,humidity,...,hour,day_of_week,device_pc1,device_pc2,device_pc3,device_pc4,direction_uplink,measured_qos_delay,measurement,operator
0,52.513400,13.334100,2.175680,0.677445,0.540301,34.177383,0.022623,0.010474,23.611542,0.577243,...,19,2,1,0,0,0,0,1,4,0
1,52.511551,13.322817,1.709962,0.694661,0.459718,34.568256,0.022329,0.010960,23.523195,0.579574,...,19,2,1,0,0,0,0,1,4,0
2,52.511688,13.323017,1.284057,0.718442,0.401192,34.480286,0.021315,0.010883,23.510515,0.579784,...,19,2,1,0,0,0,0,1,4,0
3,52.511795,13.323330,0.974575,0.737099,0.352550,34.467857,0.020580,0.010953,23.493822,0.579999,...,19,2,1,0,0,0,0,1,4,0
4,52.511730,13.322922,0.829943,0.751198,0.310703,34.601536,0.020108,0.011040,23.477451,0.580176,...,19,2,1,0,0,0,0,1,4,0


In [11]:
# Test 3: Formatted output
print("\n--- Formatted output ---")
if gen1 is not None:
    formatted = format_output(gen1)
    print(f"\nColumns ({len(formatted.columns)}):")
    print(list(formatted.columns))
    print(f"\n{formatted.to_string()}")




--- Formatted output ---

Columns (42):
['timestamp', 'device', 'Latitude', 'Longitude', 'speed_kmh', 'Altitude', 'precipIntensity', 'precipProbability', 'temperature', 'humidity', 'windSpeed', 'Traffic Jam Factor', 'Traffic Distance', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max', 'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz', 'PCell_Downlink_frequency', 'PCell_Band_Indicator', 'PCell_Downlink_bandwidth_MHz', 'PCell_Uplink_bandwidth_MHz', 'PCell_Downlink_Average_MCS', 'PCell_Downlink_Num_RBs', 'PCell_Downlink_TB_Size', 'PCell_DL_RBs_MCS_Low', 'PCell_DL_RBs_MCS_Mid', 'PCell_DL_RBs_MCS_High', 'PCell_Uplink_Num_RBs', 'PCell_Uplink_TB_Size', 'PCell_Uplink_Tx_Power_(dBm)', 'datarate', 'Pos in Ref Round', 'target_datarate', 'ping_ms', 'COG', 'jitter', 'direction_uplink', 'measured_qos_delay', 'measurement', 'operator']

                  timestamp device   Latitude  Longitude  speed_kmh   Altitude  precipIntensity  precipProbability  temperature  humidity  windSpeed  Traffic Jam Fa

In [12]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler


In [13]:
# ============================================================
# 2. BUILD SEGMENTS — ALL 4 DEVICES
# ============================================================
WINDOW_SIZE = 5
DISC_FEATURES = [
    'speed_kmh', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
    'PCell_SNR_1', 'PCell_SNR_2', 'PCell_freq_MHz',
    'PCell_Downlink_frequency', 'PCell_Band_Indicator',
    'datarate', 'ping_ms',
]

all_segments = []
device_cols = ['device_pc1', 'device_pc2', 'device_pc3', 'device_pc4']

for d_col in device_cols:
    device_id = d_col.replace('device_', '')
    sub = df[df[d_col] == 1].sort_values('timestamp').reset_index(drop=True)
    gaps = sub['timestamp'].diff().dt.total_seconds()
    break_points = gaps[gaps > 60].index.tolist()
    starts = [0] + break_points
    ends = break_points + [len(sub)]

    for s, e in zip(starts, ends):
        seg = sub.iloc[s:e].reset_index(drop=True)
        if len(seg) >= SEQ_LEN + WINDOW_SIZE + 5:
            all_segments.append({'device': device_id, 'data': seg})

print(f"Total segments across all devices: {len(all_segments)}")
for device_id in ['pc1', 'pc2', 'pc3', 'pc4']:
    count = sum(1 for s in all_segments if s['device'] == device_id)
    rows = sum(len(s['data']) for s in all_segments if s['device'] == device_id)
    print(f"  {device_id}: {count} segments, {rows} rows")

Total segments across all devices: 64
  pc1: 19 segments, 59519 rows
  pc2: 14 segments, 42879 rows
  pc3: 18 segments, 59723 rows
  pc4: 13 segments, 42821 rows


In [14]:

# ============================================================
# 3. SEGMENT-LEVEL SPLIT (prevents leakage)
# ============================================================
np.random.seed(42)
shuffled_segs = all_segments.copy()
np.random.shuffle(shuffled_segs)

n_seg = len(shuffled_segs)
n_train_seg = int(n_seg * 0.7)
n_val_seg = int(n_seg * 0.15)

train_segs = shuffled_segs[:n_train_seg]
val_segs = shuffled_segs[n_train_seg:n_train_seg + n_val_seg]
test_segs = shuffled_segs[n_train_seg + n_val_seg:]

print(f"\nSegment split: Train={len(train_segs)}, Val={len(val_segs)}, Test={len(test_segs)}")


Segment split: Train=44, Val=9, Test=11


In [15]:
# ============================================================
# HARDER DISCRIMINATOR TASK: raw values instead of diffs,
# shorter windows, added noise to both real and generated
# ============================================================

generate_fn = generate_direct_lstm
 # shorter windows = less temporal signal to exploit

# Use RAW values, not diffs — removes the autocorrelation signal
def build_windows_raw(segments, split_name):
    """Build windows using raw scaled values, not differences."""
    real_w = []
    for seg_info in segments:
        seg = seg_info['data']
        for i in range(0, len(seg) - WINDOW_SIZE, WINDOW_SIZE):
            window = seg.iloc[i:i + WINDOW_SIZE]
            vals = window[DISC_FEATURES].values.astype(np.float32)
            real_w.append(vals)

    sample_points = []
    for seg_info in segments:
        seg = seg_info['data']
        device_id = seg_info['device']
        for i in range(SEQ_LEN + 5, len(seg) - WINDOW_SIZE - 5, WINDOW_SIZE):
            sample_points.append((seg, i, device_id))
    np.random.shuffle(sample_points)

    n_target = len(real_w)
    gen_w = []
    for sp_idx, (seg, idx, device_id) in enumerate(sample_points):
        if len(gen_w) >= n_target:
            break
        row = seg.iloc[idx]
        ts = str(row['timestamp'])
        gen = generate_fn(
            device_id, ts, n_steps=WINDOW_SIZE, mode='with_gps',
            lat=row['Latitude'], lon=row['Longitude']
        )
        if gen is not None and len(gen) == WINDOW_SIZE:
            available = [c for c in DISC_FEATURES if c in gen.columns]
            if len(available) == len(DISC_FEATURES):
                vals = gen[DISC_FEATURES].values.astype(np.float32)
                gen_w.append(vals)

    n = min(len(real_w), len(gen_w))
    real_w = real_w[:n]
    gen_w = gen_w[:n]
    print(f"  {split_name}: {n} real + {n} generated = {2*n}")
    return real_w, gen_w


# Rebuild with raw values
print("Building RAW windows (no diffs)...")
train_real, train_gen = build_windows_raw(train_segs, "Train")
val_real, val_gen = build_windows_raw(val_segs, "Val")
test_real, test_gen = build_windows_raw(test_segs, "Test")


Building RAW windows (no diffs)...
  Train: 28013 real + 28013 generated = 56026
  Val: 5528 real + 5528 generated = 11056
  Test: 7023 real + 7023 generated = 14046


In [17]:


# ============================================================
# 5. ASSEMBLE ARRAYS + SCALE (fit on train only)
# ============================================================
def assemble(real_w, gen_w):
    X = np.concatenate([np.array(real_w), np.array(gen_w)], axis=0)
    y = np.concatenate([
        np.ones((len(real_w), 1), dtype=np.float32),
        np.zeros((len(gen_w), 1), dtype=np.float32)
    ], axis=0)
    perm = np.random.permutation(len(X))
    return X[perm], y[perm]



X_train, y_train = assemble(train_real, train_gen)
X_val, y_val = assemble(val_real, val_gen)
X_test, y_test = assemble(test_real, test_gen)

# Fit scaler on train only
n_feat = X_train.shape[2]
disc_scaler = StandardScaler()
disc_scaler.fit(X_train.reshape(-1, n_feat))

def apply_scaler(X, ws):
    flat = disc_scaler.transform(X.reshape(-1, n_feat))
    return flat.reshape(-1, ws, n_feat)

X_train = apply_scaler(X_train, WINDOW_SIZE)
X_val = apply_scaler(X_val, WINDOW_SIZE)
X_test = apply_scaler(X_test, WINDOW_SIZE)

print(f"Final shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

# Use ONLY continuous features (drop the 3 discrete ones that are already fine)
CONT_FEATURES = [
    'speed_kmh', 'PCell_RSRP_max', 'PCell_RSRQ_max', 'PCell_RSSI_max',
    'PCell_SNR_1', 'PCell_SNR_2', 'datarate', 'ping_ms',
]
cont_idx = [DISC_FEATURES.index(f) for f in CONT_FEATURES]
X_train = X_train[:, :, cont_idx]
X_val = X_val[:, :, cont_idx]
X_test = X_test[:, :, cont_idx]
n_feat = len(cont_idx)

# DataLoaders
train_ds = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
val_ds = TensorDataset(torch.FloatTensor(X_val), torch.FloatTensor(y_val))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)










Final shapes: Train=(56026, 5, 11), Val=(11056, 5, 11), Test=(14046, 5, 11)


In [18]:
# ============================================================
# 1. DISCRIMINATOR MODEL (logits output for stability)
# ============================================================
class Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.head_dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)   # <-- no sigmoid; return logits
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.head_dropout(out[:, -1, :])
        return self.classifier(out)   # logits


In [19]:


# ============================================================
# 6. TRAIN DISCRIMINATOR (stabilized)
# ============================================================
print("\n" + "="*60)
print("TRAINING DISCRIMINATOR")
print("="*60)

disc = Discriminator(
    input_dim=n_feat,
    hidden_dim=64,
    num_layers=1,
    dropout=0.3
).to(device_torch)

# Sensible LR + meaningful weight decay
BASE_LR = 3e-4
optimizer = torch.optim.AdamW(
    disc.parameters(),
    lr=BASE_LR,
    weight_decay=1e-4,
    betas=(0.9, 0.999),
    eps=1e-8,
)

# BCEWithLogitsLoss: more numerically stable than Sigmoid + BCELoss
criterion = nn.BCEWithLogitsLoss()

# Warmup + cosine schedule — smooth LR curve, no plateau oscillation
MAX_EPOCHS = 50
WARMUP_EPOCHS = 3
steps_per_epoch = len(train_loader)
total_steps = MAX_EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Parameters: {sum(p.numel() for p in disc.parameters())}")
print(f"Base LR: {BASE_LR}, Warmup: {WARMUP_EPOCHS} epochs, Total: {MAX_EPOCHS}")

best_val_loss = float('inf')
best_val_acc = 0.0
best_state = None
best_epoch = 0
patience = 0
MAX_PATIENCE = 10      # a touch more forgiving, since LR schedule is smoother
NOISE_STD = 0.05

# Rolling history for trend reporting
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}

for epoch in range(MAX_EPOCHS):
    # ---- Train ----
    disc.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device_torch)
        y_batch = y_batch.to(device_torch)

        # Input noise (regularization)
        X_noisy = X_batch + NOISE_STD * torch.randn_like(X_batch)

        optimizer.zero_grad()
        logits = disc(X_noisy)
        loss = criterion(logits, y_batch)

        # Skip update if loss goes non-finite (rare but kills runs)
        if not torch.isfinite(loss):
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(disc.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()   # per-step scheduler

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            train_correct += ((probs > 0.5).float() == y_batch).sum().item()
        train_loss += loss.item() * len(X_batch)
        train_total += len(X_batch)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # ---- Val ----
    disc.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device_torch)
            y_batch = y_batch.to(device_torch)
            logits = disc(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * len(X_batch)
            probs = torch.sigmoid(logits)
            val_correct += ((probs > 0.5).float() == y_batch).sum().item()
            val_total += len(X_batch)

    val_loss /= val_total
    val_acc = val_correct / val_total
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # Track best by val LOSS (smoother than accuracy on small val sets)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        best_state = {k: v.clone() for k, v in disc.state_dict().items()}
        best_epoch = epoch + 1
        patience = 0
        marker = " *"
    else:
        patience += 1
        marker = ""

    print(
        f"  Epoch {epoch+1:3d} | "
        f"LR: {current_lr:.2e} | "
        f"Train L: {train_loss:.4f} A: {train_acc:.3f} | "
        f"Val L: {val_loss:.4f} A: {val_acc:.3f} | "
        f"Pat: {patience}{marker}"
    )

    if patience >= MAX_PATIENCE:
        print(f"  Early stop at epoch {epoch+1}")
        break

print(f"\nBest model: epoch {best_epoch} | Val Loss: {best_val_loss:.4f} | Val Acc: {best_val_acc:.3f}")
disc.load_state_dict(best_state)
disc.eval()


TRAINING DISCRIMINATOR
Parameters: 21057
Base LR: 0.0003, Warmup: 3 epochs, Total: 50
  Epoch   1 | LR: 1.00e-04 | Train L: 0.6929 A: 0.503 | Val L: 0.6929 A: 0.499 | Pat: 0 *
  Epoch   2 | LR: 2.00e-04 | Train L: 0.6546 A: 0.622 | Val L: 0.5693 A: 0.714 | Pat: 0 *
  Epoch   3 | LR: 3.00e-04 | Train L: 0.4961 A: 0.771 | Val L: 0.4967 A: 0.774 | Pat: 0 *
  Epoch   4 | LR: 3.00e-04 | Train L: 0.4285 A: 0.815 | Val L: 0.4469 A: 0.801 | Pat: 0 *
  Epoch   5 | LR: 2.99e-04 | Train L: 0.3932 A: 0.835 | Val L: 0.4027 A: 0.822 | Pat: 0 *
  Epoch   6 | LR: 2.97e-04 | Train L: 0.3611 A: 0.850 | Val L: 0.3886 A: 0.830 | Pat: 0 *
  Epoch   7 | LR: 2.95e-04 | Train L: 0.3245 A: 0.868 | Val L: 0.3473 A: 0.847 | Pat: 0 *
  Epoch   8 | LR: 2.92e-04 | Train L: 0.3022 A: 0.878 | Val L: 0.3216 A: 0.861 | Pat: 0 *
  Epoch   9 | LR: 2.88e-04 | Train L: 0.2855 A: 0.886 | Val L: 0.2996 A: 0.872 | Pat: 0 *
  Epoch  10 | LR: 2.84e-04 | Train L: 0.2735 A: 0.891 | Val L: 0.3198 A: 0.863 | Pat: 1
  Epoch  11 | L

Discriminator(
  (lstm): LSTM(8, 64, batch_first=True)
  (head_dropout): Dropout(p=0.3, inplace=False)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [20]:
# ============================================================
# 7. EVALUATE ON TEST SET
# ============================================================
print("\n" + "="*60)
print("DISCRIMINATOR RESULTS")
print("="*60)

X_test_tensor = torch.FloatTensor(X_test).to(device_torch)
with torch.no_grad():
    logits = disc(X_test_tensor)
    pred_probs = torch.sigmoid(logits).cpu().numpy()

pred_labels = (pred_probs > 0.5).astype(int)
true_labels = y_test.astype(int)

accuracy = accuracy_score(true_labels, pred_labels)
print(f"\nTest Accuracy: {accuracy*100:.1f}%")
print(f"Best Val Accuracy: {best_val_acc*100:.1f}%")

print(f"\nInterpretation:")
if accuracy < 0.55:
    print(f"  EXCELLENT — Cannot distinguish real from generated")
elif accuracy < 0.65:
    print(f"  GOOD — Barely distinguishable")
elif accuracy < 0.75:
    print(f"  MODERATE — Some differences detected")
elif accuracy < 0.85:
    print(f"  FAIR — Noticeable differences")
else:
    print(f"  POOR — Easily distinguishable")

print(f"\nClassification Report:")
print(classification_report(true_labels, pred_labels,
                            target_names=['Generated', 'Real']))

print(f"\nConfusion Matrix:")
cm = confusion_matrix(true_labels, pred_labels)
print(f"  Predicted:  Generated  Real")
print(f"  Generated:  {cm[0][0]:8d}  {cm[0][1]:4d}")
print(f"  Real:       {cm[1][0]:8d}  {cm[1][1]:4d}")

# Average discriminator score
print(f"\nDiscriminator Scores:")
real_mask = y_test.flatten() == 1
gen_mask = y_test.flatten() == 0
print(f"  Real data avg score:      {pred_probs[real_mask].mean():.4f} (ideal: 1.0)")
print(f"  Generated data avg score: {pred_probs[gen_mask].mean():.4f} (ideal near 0.5 = fooling discriminator)")


DISCRIMINATOR RESULTS

Test Accuracy: 94.4%
Best Val Accuracy: 91.7%

Interpretation:
  POOR — Easily distinguishable

Classification Report:
              precision    recall  f1-score   support

   Generated       0.94      0.94      0.94      7023
        Real       0.94      0.94      0.94      7023

    accuracy                           0.94     14046
   macro avg       0.94      0.94      0.94     14046
weighted avg       0.94      0.94      0.94     14046


Confusion Matrix:
  Predicted:  Generated  Real
  Generated:      6636   387
  Real:            402  6621

Discriminator Scores:
  Real data avg score:      0.9232 (ideal: 1.0)
  Generated data avg score: 0.0781 (ideal near 0.5 = fooling discriminator)


In [21]:
# ============================================================
# FEATURE ABLATION — raw value version
# ============================================================
print("\n" + "="*60)
print("FEATURE ABLATION ANALYSIS (RAW VALUES)")
print("="*60)

disc.eval()
X_test_t = torch.FloatTensor(X_test).to(device_torch)
y_test_flat = y_test.astype(int).flatten()

with torch.no_grad():
    base_probs = torch.sigmoid(disc(X_test_t)).cpu().numpy().flatten()
base_acc = accuracy_score(y_test_flat, (base_probs > 0.5).astype(int))
print(f"Baseline accuracy: {base_acc*100:.2f}%\n")

results = []
for f_idx, feat_name in enumerate(CONT_FEATURES):
    X_ablated = X_test.copy()
    X_ablated[:, :, f_idx] = X_ablated[:, :, f_idx].mean()
    with torch.no_grad():
        probs = torch.sigmoid(disc(torch.FloatTensor(X_ablated).to(device_torch))).cpu().numpy().flatten()
    acc = accuracy_score(y_test_flat, (probs > 0.5).astype(int))
    drop = base_acc - acc
    results.append((feat_name, acc, drop))

results.sort(key=lambda x: -x[2])
print(f"{'Feature':<25} {'Ablated Acc':>12} {'Drop':>8}")
print("-" * 47)
for name, acc, drop in results:
    print(f"{name:<25} {acc*100:>10.2f}%  {drop*100:>6.2f}%")

# ============================================================
# PER-FEATURE STATS (raw values, not diffs)
# ============================================================
print(f"\n{'Feature':<25} {'R_mean':>8} {'G_mean':>8} {'R_std':>8} {'G_std':>8} {'std_ratio':>10}")
print("-" * 70)
for f_idx, name in enumerate(CONT_FEATURES):
    real = X_test[y_test_flat == 1][:, :, f_idx].flatten()
    gen  = X_test[y_test_flat == 0][:, :, f_idx].flatten()
    ratio = gen.std() / real.std() if real.std() > 0 else 0
    print(f"{name:<25} {real.mean():+8.3f} {gen.mean():+8.3f} "
          f"{real.std():8.3f} {gen.std():8.3f} {ratio:10.2f}x")

# ============================================================
# WITHIN-WINDOW VARIANCE (key signal for "too smooth")
# ============================================================
print(f"\nWithin-window std (measures per-window variability):")
print(f"{'Feature':<25} {'R_winstd':>10} {'G_winstd':>10} {'ratio':>8}")
print("-" * 55)
for f_idx, name in enumerate(CONT_FEATURES):
    real_wins = X_test[y_test_flat == 1][:, :, f_idx]  # (n, window_size)
    gen_wins  = X_test[y_test_flat == 0][:, :, f_idx]
    r_std = np.mean(np.std(real_wins, axis=1))
    g_std = np.mean(np.std(gen_wins, axis=1))
    ratio = g_std / r_std if r_std > 0 else 0
    print(f"{name:<25} {r_std:10.4f} {g_std:10.4f} {ratio:7.2f}x")


FEATURE ABLATION ANALYSIS (RAW VALUES)
Baseline accuracy: 94.38%

Feature                    Ablated Acc     Drop
-----------------------------------------------
PCell_RSRP_max                 76.39%   17.99%
PCell_SNR_2                    79.49%   14.89%
speed_kmh                      82.88%   11.51%
PCell_RSSI_max                 83.09%   11.29%
datarate                       85.04%    9.34%
PCell_SNR_1                    85.31%    9.07%
ping_ms                        87.61%    6.78%
PCell_RSRQ_max                 89.83%    4.56%

Feature                     R_mean   G_mean    R_std    G_std  std_ratio
----------------------------------------------------------------------
speed_kmh                   +0.037   -0.050    1.127    0.814       0.72x
PCell_RSRP_max              -0.078   +0.056    1.064    0.908       0.85x
PCell_RSRQ_max              -0.001   +0.141    1.138    0.871       0.77x
PCell_RSSI_max              +0.061   +0.071    1.065    0.941       0.88x
PCell_SNR_1         